# Historical BTS flight-data exploration (January 2024)

This notebook reads only pipeline outputs made from the official BTS CSV. It does not create sample or synthetic production data. Download and process the dataset using the commands in the repository README before running these cells. The pipeline scans and writes in chunks so raw CSVs are not loaded into notebook memory.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
EDA = ROOT / "reports" / "eda" / "2024-01"
QUALITY_REPORT = ROOT / "docs" / "data_quality_report.md"
if not (EDA / "overview.csv").exists():
    raise FileNotFoundError(
        "No measured BTS EDA outputs found. Run the BTS January 2024 ingestion pipeline first."
    )

## Dataset quality and totals

The overview and report are computed from the actual source file. Missing values are not imputed. Cancellation and arrival-delay denominators are kept explicit.

In [ ]:
overview = pd.read_csv(EDA / "overview.csv")
display(overview)
print(QUALITY_REPORT.read_text(encoding="utf-8"))

## Airline and arrival-airport delays

Delay percentages use flights with known arrival delay that were neither cancelled nor diverted. The airport table groups by destination airport.

In [ ]:
airline = pd.read_csv(EDA / "airline_delays.csv")
airport = pd.read_csv(EDA / "airport_delays.csv")
display(airline.sort_values("delay_percentage", ascending=False).head(20))
display(airport.sort_values("delay_percentage", ascending=False).head(20))
display(Image(filename=str(EDA / "airline_delay_rates.png")))
display(Image(filename=str(EDA / "airport_delay_rates.png")))

## Routes, scheduled departure hour, and arrival-delay distribution

In [ ]:
routes = pd.read_csv(EDA / "route_delays.csv")
hours = pd.read_csv(EDA / "hour_delays.csv")
delay_distribution = pd.read_csv(EDA / "arrival_delay_distribution.csv")
display(routes.sort_values("delay_percentage", ascending=False).head(20))
display(hours.sort_values("departure_hour").head(24))
display(delay_distribution)
for name in (
    "route_delay_rates.png",
    "hourly_delay_rates.png",
    "arrival_delay_distribution.png",
):
    display(Image(filename=str(EDA / name)))

## Interpretation notes

These are descriptive statistics for a single month and are not forecasts or causal estimates. BTS reporting coverage, missing delay values, cancellations, diversions, and route/airline sample sizes affect comparisons. The generated tables include flight and eligible-delay counts so percentages can be interpreted with their denominators.